# OGC API – Features from Python (T070)

Reads all six collections of the butterfly-monitoring pygeoapi with `requests` + `GeoDataFrame.from_features` (paging with `sortby=id`), then once through GDAL's OAPIF driver with `geopandas.read_file`.

The token is read from the header file `~/.ogc-header` (`Authorization: Token <key>`, see `docs/setup/ogc-access.md`); optional `OGC_HEADER_FILE` and `OGC_BASE` override the defaults. No environment variable holds the key.

In [2]:
import os
import geopandas as gpd
import matplotlib.pyplot as plt
import requests

BASE = os.environ.get("OGC_BASE", "http://localhost:8080/ogc")
# The credential lives in one file (chmod 600): "Authorization: Token <key>". Nothing is read from the environment.
HEADER_FILE = os.path.expanduser(os.environ.get("OGC_HEADER_FILE", "~/.ogc-header"))
with open(HEADER_FILE) as f:
    headers = {"Authorization": f.read().strip().split(":", 1)[1].strip()}  # never print this
COLLECTIONS = [
    "transects",
    "transect-sections",
    "butterfly-observations",
    "plant-resources",
    "habitat-features",
    "observation-hexgrid",
]
print("geopandas", gpd.__version__, "| base", BASE)

geopandas 1.2.0 | base http://localhost:8080/ogc


In [3]:
def fetch_all(collection, limit=1000):
    """Page through /items with sortby=id, following rel=next. Returns (GeoDataFrame, pages, numberMatched)."""
    url = f"{BASE}/collections/{collection}/items"
    params = {"f": "json", "limit": limit, "sortby": "id"}
    features, pages, matched = [], 0, None
    while url:
        r = requests.get(url, params=params, headers=headers, timeout=60)
        r.raise_for_status()
        body = r.json()
        pages += 1
        features += body["features"]
        matched = body.get("numberMatched", matched)
        url = next((l["href"] for l in body["links"] if l["rel"] == "next"), None)
        params = None  # the next link already carries every parameter
    return gpd.GeoDataFrame.from_features(features, crs="EPSG:4326"), pages, matched

In [4]:
import pandas as pd

frames, rows = {}, []
for c in COLLECTIONS:
    gdf, pages, matched = fetch_all(c)
    assert len(gdf) == matched, f"{c}: read {len(gdf)} but numberMatched {matched}"
    frames[c] = gdf
    rows.append({
        "collection": c,
        "features read": len(gdf),
        "numberMatched": matched,
        "pages": pages,
        "geometry types": ", ".join(sorted(gdf.geom_type.dropna().unique())),
        "columns": len(gdf.columns),
    })
assert next(r for r in rows if r["collection"] == "butterfly-observations")["pages"] > 1
pd.DataFrame(rows)

,collection,features read,numberMatched,pages,geometry types,columns
0,transects,33,33,1,LineString,13
1,transect-sections,207,207,1,LineString,15
2,butterfly-observations,3982,3982,4,Point,26
3,plant-resources,112,112,1,"Point, Polygon",15
4,habitat-features,83,83,1,"LineString, Point, Polygon",17
5,observation-hexgrid,454,454,1,Polygon,11


In [5]:
for c, gdf in frames.items():
    print(c)
    print(gdf.drop(columns="geometry").dtypes.to_string())
    print()

transects
id                     str
project_id             str
project_name           str
code                   str
name                   str
description            str
target_length_m    float64
length_m           float64
habitat_summary        str
active                bool
created_at             str
section_count        int64

transect-sections
id                             str
transect_id                    str
transect_code                  str
section_code                   str
sequence_number              int64
habitat_type_code              str
habitat_type_name_en           str
management_type_code           str
management_type_name_en        str
length_m                   float64
notes                          str
created_at                     str
start_m                    float64
end_m                      float64

butterfly-observations
id                               str
project_id                       str
project_name                     str
survey_visit_id       

In [ ]:
# Second read path: GDAL OAPIF driver through geopandas.read_file. GDAL_HTTP_HEADERS is ignored by the
# driver (D-080), so the header comes from a file named in GDAL_HTTP_HEADER_FILE.
os.environ["GDAL_HTTP_HEADER_FILE"] = HEADER_FILE
via_gdal = gpd.read_file(f"OAPIF:{BASE}", layer="transects")
print(len(via_gdal), "transects via GDAL;", len(frames["transects"]), "via requests")
via_gdal.drop(columns="geometry").head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
frames["transects"].plot(ax=ax, color="black", linewidth=1.5, label="transects")
frames["butterfly-observations"].plot(ax=ax, color="tab:orange", markersize=2, alpha=0.4, label="observations")
ax.set_title("Transects and butterfly observations (EPSG:4326)")
ax.legend()
plt.show()